# Lab 1

**Name:** Alan Nur

**Due Date:** 8 Apr 2026

## 1. Return Slips

Done

## 2. Docker Files code



In [ ]:
ARG CUDA_VERSION=13.1.0 #G14 5060
#ARG CUDA_VERSION=13.0.0 #4090

ARG OS_VERSION=22.04

### build image ###
FROM nvidia/cuda:${CUDA_VERSION}-base-ubuntu${OS_VERSION} AS build

# Lecture code reference: Env vars for the nvidia-container-runtime.
ENV NVIDIA_VISIBLE_DEVICES=all
ENV NVIDIA_DRIVER_CAPABILITIES=graphics,utility,compute
ENV QT_X11_NO_MITSHM=1
ENV DEBIAN_FRONTEND=noninteractive

RUN echo 'Acquire::Retries "3";' > /etc/apt/apt.conf.d/80retries

# Install basic setup tools and ROS 2 repository
RUN apt-get update && apt-get install -y --no-install-recommends \
    software-properties-common \
    locales \
    curl \
    gnupg2 \
    lsb-release \
    && locale-gen en_US en_US.UTF-8 \
    && update-locale LC_ALL=en_US.UTF-8 LANG=en_US.UTF-8 \
    && rm -rf /var/lib/apt/lists/*

ENV LANG=en_US.UTF-8

# Add ROS 2 GPG key and repo
RUN curl -sSL https://raw.githubusercontent.com/ros/rosdistro/master/ros.key -o /usr/share/keyrings/ros-archive-keyring.gpg && \
    echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] http://packages.ros.org/ros2/ubuntu $(lsb_release -cs) main" | tee /etc/apt/sources.list.d/ros2.list > /dev/null

# Lecture code reference: Install ROS 2 Humble Desktop and other apt packages from the screenshots
RUN apt-get update && apt-get install -y --no-install-recommends \
    ros-humble-desktop \
    bash-completion \
    build-essential \
    cmake \
    gdb \
    git \
    openssh-client \
    python3-argcomplete \
    python3-pip \
    python-is-python3 \
    ros-dev-tools \
    ros-humble-ament-* \
    ros-humble-moveit \
    ros-humble-realsense2-* \
    ros-humble-usb-cam \
    ros-humble-topic-tools \
    vim \
    && rm -rf /var/lib/apt/lists/*

# Lecture code reference: Install pyrealsense2
RUN pip install pyrealsense2

# Install lerobot system dependencies based on its dockerfile
RUN apt-get update && apt-get install -y --no-install-recommends \
    libglib2.0-0 libegl1-mesa-dev ffmpeg \
    libusb-1.0-0-dev speech-dispatcher libgeos-dev portaudio19-dev \
    pkg-config ninja-build \
    && rm -rf /var/lib/apt/lists/*

# For devcontainers, copy the local project folder to install lerobot.
# (If mounting a volume over this later, you will still have the installed dependencies)
COPY third_party/lerobot /tmp/lerobot
RUN pip install -e /tmp/lerobot

### dev ###
FROM build AS dev 

# Lecture code reference: change user
ARG USERNAME=ubuntu
ARG USER_UID=1000
ARG USER_GID=$USER_UID
RUN if ! id -u $USER_UID >/dev/null 2>&1; then \
        groupadd --gid $USER_GID $USERNAME && \
        useradd -s /bin/bash --uid $USER_UID --gid $USER_GID -m $USERNAME; \
    fi

# Lecture code reference: Add sudo support for the non-root user
RUN apt-get update && \
    apt-get install -y sudo && \
    echo "$USERNAME ALL=(root) NOPASSWD:ALL" > /etc/sudoers.d/$USERNAME && \
    chmod 0440 /etc/sudoers.d/$USERNAME

# Lecture code reference: Add user to video group to allow access to webcam
RUN sudo usermod --append --groups video,plugdev $USERNAME

# Switch from root to user
USER $USERNAME

# Lecture code reference: general dev tools
RUN sudo apt-get update && sudo apt-get install -y --no-install-recommends \
    vim \
    nano \
    less \
    tmux \
    && sudo rm -rf /var/lib/apt/lists/*

# Lecture code reference: environment settings in the bashrc
RUN echo "export PATH=\"\$HOME/.local/bin:\$PATH\"" >> /home/ubuntu/.bashrc && \
    echo "source /opt/ros/humble/setup.bash" >> /home/ubuntu/.bashrc  && \
    echo "source /home/ubuntu/techin517/ros2_ws/install/setup.bash" >> /home/ubuntu/.bashrc && \
    echo "export ROS_LOCALHOST_ONLY=1" >> /home/ubuntu/.bashrc && \
    echo "export HF_USER=ubuntu" >> /home/ubuntu/.bashrc && \
    echo "export XDG_RUNTIME_DIR=/tmp/runtime-ubuntu" >> /home/ubuntu/.bashrc

# Default shell
CMD ["/bin/bash"]

## 3. Calibration Files


[Link](https://drive.google.com/drive/folders/1G1ait1DY4dQ1Uyc1nCgexFtpGHqKMeg0)


## 4. Video Inference
[Link](https://drive.google.com/drive/folders/1ajTAS3wzlYgiZnkoHM6pdgEXU6S3VawJ?usp=sharing)


## 5. The trade-offs of using VLAs.
What are they good for?
I find these models great because they adapt easily. Instead of programming every single move, the robot can hear a new instruction and just do it. Seeing the world and understanding speech at the same time makes them much better than older methods.

What are they bad at?
Their biggest flaws are being slow and fragile. They take a long time to process information. If a room looks even slightly different from what they know, they fail quickly. They also mess up long tasks because tiny mistakes add up over time. Even simple tasks like picking things up still fail too often.

How might you mitigate shortcomings?
A few tricks help fix these issues. I can tweak the model for a new space without starting from scratch, which saves time and computer power. Splitting the brain into two parts, one fast for reactions and one slow for deep thought, fixes the speed issue. Practicing in a virtual world also helps the robot learn to fix its own mistakes instead of freezing.

What was your experience like training a policy?
For me, gathering data is the hardest part. Controlling the robot by hand is boring, and I can never record every possible situation. This makes the robot easily confused in the real world. Copying human actions is a good start, but it is not enough. The biggest frustration is that what works perfectly on a computer simulation rarely works perfectly on the actual robot.

How do you imagine these policies scaling to work in the real world?
I expect this change to happen slowly. We are already seeing these robots in organized places like factories. But for messy places like homes or hospitals, we need the computer processing to be cheaper, the safety to be much better, and a lot more practice data. Realistically, I expect them to be everywhere around 2027 or 2028.